# Problema del lago congelado

Este problema plantea que un agente debe cruzar un lago congelado, desde un extremo del mismo al otro lado. En el camino puede encontrar alguno orificios en el hielo los cuales le pueden hace caer. Utilizando el algoritmo *Q-Learning* el agente debe encontrar el camino mas óptimo. El punto de inicio se representa con *S* y por el otro lado la meta se identifica con la letra *G*.

## Paso 1. Configuración del entorno

La importación de las librerías necesarias para solucionar este problema:
- numpy
- gymnasium

In [ ]:
# Importación de librerías
import numpy as np
import gymnasium as gym
import random

In [ ]:
# Crear el entorno del lago congelado
entorno = gym.make('FrozenLake-v1', is_slippery=False, render_mode=None)

In [ ]:
# Definir los estados y acciones de nuestro ambiente
no_estados = entorno.observation_space.n
no_acciones = entorno.action_space.n

In [ ]:
print(f'El número de estados es: {no_estados}')
print(f'El número de acciones es: {no_acciones}')

El número de estados es: 16
El número de acciones es: 4


## Paso 2. Definir e inicial la matriz Q

La matriz Q almacena el valor de la función Q (de la mayor ganancia o recompensa al desplazamiento del agente).

In [ ]:
# Definición e inicialización de la matriz Q
matriz_Q = np.zeros((no_estados, no_acciones))
print(matriz_Q)

[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


# Paso 3. Definición de los hiperparametros

Para que el algoritmo Q-Learning pueda entrenarse es necesario establecer ciertos parámetros, los cuales e listan a continuación:
- *learning_rate*: Determina cuanto de la información aprendida se mantiene en contexto. Su valor esta dado entre 0-1. Por ejemplo, si indico que la tasa es de 1, entonces solo considear la nueva información.
- *discount_factor*: Determina la importancia de la recompensa futura. Un valor 0 hace que el agente no vea mas allá de la recompensa actual.
- *epsilon*: la probabilidad de que el agente explore el entorno de manera aleatoria.
- *max_episodes*: El número total de veces en que el agente intenta resolver el problema.
- *max_steps_per_episode*: El número máximo de acciones que el agente puede tomar en un episodio.

In [ ]:
# Definir los hiperpáremetros
tasa_aprendizaje = 0.8
factor_descuento = 0.8
epsilon = 1
max_episodios = 5000
max_pasos_por_episodio = 100



In [ ]:
# Definición de recompensas por episodio
recompensa_por_episodio = []

## Paso 4. Ciclo de entrenamiento

En este ciclo, se estará leyendo un estado de entrada y una posible accion por cada paso y con respecto a la función se generará un valor de recompensa.

In [ ]:
# Hiperparámetros de epsilon
min_epsilon = 0.01
epsilon_decay_rate = 0.0001

# Definición del ciclo de entrenamiento
for episodio in range(max_episodios):
    # Resetear el entorno
    estado, info = entorno.reset()
    terminado = False
    inconcluso = False
    recompensa_actual = 0

    for paso in range(max_pasos_por_episodio):
      # 1. Elección de la acción
      if random.uniform(0, 1) < epsilon:
        accion = entorno.action_space.sample() # Acción aleatora
      else:
        accion = np.argmax(matriz_Q[estado, :]) # Acción basada en el valor conocido

      # 2. Ejecución de la acción y recepción de la recompensa y actualización del estado
      nuevo_estado, recompensa, terminado, inconcluso, info = entorno.step(accion)

      # 3. Actualizar el valor de la funcion Q(s, a) en la matriz Q con respecto de los nuevos valores
      matriz_Q[estado, accion] = matriz_Q[estado, accion] + tasa_aprendizaje * (recompensa + factor_descuento * np.max(matriz_Q[nuevo_estado, :]) - matriz_Q[estado, accion])
      estado = nuevo_estado
      recompensa_actual += recompensa
      # Verificación de no excedente de pasos
      if terminado == True:
        break

    # Decremento de epsilon
    epsilon = max(min_epsilon, epsilon - epsilon_decay_rate)
    # Almacenamiento de las recompensas por episodio
    recompensa_por_episodio.append(recompensa_actual)

    # Retroalimentación del avance de los episodios
    if episodio % 1000 == 0:
      print(f'Episodio: {episodio}, recompensa promedio: {np.mean(recompensa_por_episodio[-100:])}, epsilon: {epsilon}')
entorno.close()
print('Entrenando terminado')



Episodio: 0, recompensa promedio: 0.0, epsilon: 0.9999
Episodio: 1000, recompensa promedio: 0.03, epsilon: 0.899900000000011
Episodio: 2000, recompensa promedio: 0.07, epsilon: 0.799900000000022
Episodio: 3000, recompensa promedio: 0.21, epsilon: 0.699900000000033
Episodio: 4000, recompensa promedio: 0.24, epsilon: 0.5999000000000441
Entrenando terminado


In [ ]:
print(matriz_Q)

[[0.262144 0.32768  0.32768  0.262144]
 [0.262144 0.       0.4096   0.32768 ]
 [0.32768  0.512    0.32768  0.4096  ]
 [0.4096   0.       0.32768  0.32768 ]
 [0.32768  0.4096   0.       0.262144]
 [0.       0.       0.       0.      ]
 [0.       0.64     0.       0.4096  ]
 [0.       0.       0.       0.      ]
 [0.4096   0.       0.512    0.32768 ]
 [0.4096   0.64     0.64     0.      ]
 [0.512    0.8      0.       0.512   ]
 [0.       0.       0.       0.      ]
 [0.       0.       0.       0.      ]
 [0.       0.64     0.8      0.512   ]
 [0.64     0.8      1.       0.64    ]
 [0.       0.       0.       0.      ]]


## Paso 5. Evaluación del agente entrenado

Para evaluar el entrenamiendo del agente, es necesario ejecutar algunos episodios adicionales pero ahora empleando las politicas ya calculadas (matriz_Q) sin necesidad de explorar nuevamente el espacio de estados.

In [ ]:
# Evaluando el rendimiento del agente

episodios_prueba = 100
exitos = 0

# Recrear el entorno para evaluación
entorno_eval = gym.make('FrozenLake-v1', is_slippery=False, render_mode='rgb_array')

print("\nEvaluando el desempeño del agente")

for episodio in range(episodios_prueba):
  estado, info = entorno_eval.reset()
  terminado = False
  inconcluso = False

  for paso in range(max_pasos_por_episodio):
    accion = np.argmax(matriz_Q[estado, :])
    nuevo_estado, recompensa, terminado, inconcluso, info = entorno_eval.step(accion)
    estado = nuevo_estado
    if terminado and recompensa == 1:
      exitos += 1
      break
    elif terminado or inconcluso:
      break
  entorno_eval.close()

tasa_exito = (exitos / episodios_prueba) * 100
print(f'Tasa de éxito del agente: {tasa_exito}%')


Evaluando el desempeño del agente
Tasa de éxito del agente: 100.0%
